# ***K1–K16 data quality native noise, outliers, and all 12 prediction targets***

Select `SAMPLING_INTERVAL_SECONDS = 5` or `60` in Import. The native profile inspects **five-second sensor files and ten-second MPPT files** at their own cadences; MPPT is never upsampled. The minute profile inspects the provided minute aggregates. Data remain unchanged.

Raw timestamps identify readings; minute timestamps identify bin starts, with means and summed doses available after bin end. Sampling cadence, prediction duration, and controller update interval are separate choices. EDA does not establish which cadence predicts best.

Files are inspected sequentially. Retained reports contain exact full-file statistics, timestamp indexes, and small display summaries rather than every full raw table. Display resampling never becomes model-training data. Configure cultivation IDs to restrict a diagnostic run.

The default is the **five-second sensor profile**. The target list retains the original nine and appends biomass estimate, BOD, and TSS. The priority group is **pH, COD, BOD, DO, and water NH3**; this is not a ranking among the five. Reports cover every configured cultivation and every target. Detailed 12-panel traces, change distributions, and spectra default to K11–K14, while cross-cultivation distributions and verdict tables cover K1–K16. Change `QUALITY_PLOT_CULTIVATIONS` to plot more.

Noise cannot be identified from variance alone: changes may be real dynamics, dosing, cleaning, quantization, or measurement error. All statistical flags are review candidates. This notebook does not delete outliers, smooth training inputs, alter source CSVs, fit a model, or choose a sampling interval using K16 outcomes. Labels derived from interpolated OD or firmware formulas are analyzed as proxy columns.

## ***Import***

In [ ]:
from pathlib import Path
import re
import sys
from io import BytesIO
import gc
import json
import platform

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

try:
    import matplotlib.pyplot as plt
    PLOTTING_AVAILABLE = True
except ImportError as error:
    PLOTTING_AVAILABLE = False
    PLOTTING_ERROR = str(error)

PROJECT_ROOT_OVERRIDE = None
SAMPLING_INTERVAL_SECONDS = 5  # 5: native sensor + 10s MPPT; 60: provided minute files
CULTIVATIONS = list(range(1, 17))
SHOW_MPPT_CHANNEL_STATISTICS = False
SHOW_LEGACY_TREND_PLOTS = False
SHOW_SENSOR_CORRELATION = False
PLOT_INTERVAL = '30min'
DETAIL_MINUTES = 15
MAX_GAP_ROWS = 8

def locate_project_root():
    if PROJECT_ROOT_OVERRIDE is not None:
        return Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'dev' / 'AGENTS.md').is_file():
            return candidate
    raise FileNotFoundError('Set PROJECT_ROOT_OVERRIDE to the project directory.')

PROJECT_ROOT = locate_project_root()
sys.path.insert(0, str(PROJECT_ROOT / 'dev/world_model'))
sys.path.insert(0, str(PROJECT_ROOT / 'dev/dataset_eda'))
import world_models as wm
import dataset_quality as dq
QUALITY_TARGETS = wm.CONTROL_TARGETS.copy()
PRIORITY_TARGETS = ['1_ph', '1_cod(mg/L)', '1_bod(mg/L)', '1_do(mg/L)', '1_nh3(mg/L)']
QUALITY_PLOT_CULTIVATIONS = [11, 12, 13, 14]
EXPORT_QUALITY_REPORTS = True
QUALITY_REPORT_DIR = PROJECT_ROOT / 'dev/dataset_eda/reports' / f'quality_{SAMPLING_INTERVAL_SECONDS}s_12targets'
QUALITY_CONFIG = {
    'priority_targets': PRIORITY_TARGETS,
    'median_window_seconds': 180,
    'mad_multiplier': 6.,
    'spike_return_fraction': .25,
    'flatline_review_minutes': 10.,
    'spike_review_pct': .1,
    'within_variance_review_pct': 20.,
    'detail_minutes': 30,
    'max_event_examples_per_target': 5,
    'spectrum_min_samples': 64,
    'spectrum_max_samples': 17280,
    'spectrum_nperseg': 4096,
    'fast_period_seconds': 60.,
    'ph_screen': [0., 14.],
    'nonnegative_targets': [t for t in QUALITY_TARGETS if t not in ['1_ph', '1_temp_air(C)']],
    'rounding_tolerance': {'biomassa_estimasi(g/L)': .000051, '1_cod(mg/L)': .051, '1_bod(mg/L)': .051, '1_nh3(mg/L)': .051},
    'default_rounding_tolerance': .0051,
    'figure_dpi': 100,
    'show_plots': True,
    'figure_dir': str(QUALITY_REPORT_DIR / 'figures') if EXPORT_QUALITY_REPORTS else None,
}
if len(QUALITY_TARGETS) != 12 or not set(PRIORITY_TARGETS).issubset(QUALITY_TARGETS):
    raise ValueError('This report must cover the configured 12-target schema and all five priorities.')
if SAMPLING_INTERVAL_SECONDS not in [5, 60]:
    raise ValueError('Choose a 5- or 60-second sensor profile.')
DATASET_DIR = PROJECT_ROOT / 'dev/dataset'
ORIGINAL_DIR = PROJECT_ROOT / 'dataset/Data Kultivasi Mikroalga K1-K16'
SOURCE_README = ORIGINAL_DIR / 'README.md'
DOSE_COLUMNS = wm.ACTIONS[:3]
HELD_COLUMNS = wm.ACTIONS[3:]
ACTION_COLUMNS = wm.ACTIONS
ESTIMATED_COLUMNS = ['od680_estimasi(AU)', 'biomassa_estimasi(g/L)',
                     'berat_kering_estimasi(g)', 'berat_basah_estimasi(g)']
PROCESS_PLOT_COLUMNS = ['1_ph', '1_do(mg/L)', '1_temp_air(C)', '1_cod(mg/L)']
FILE_PATTERN = re.compile(r'^K([0-9]{2})_(sensor|rasp_mppt)_(menit|5detik|10detik)[.]csv$')
SUFFIXES = {'sensor': '5detik' if SAMPLING_INTERVAL_SECONDS == 5 else 'menit',
            'rasp_mppt': '10detik' if SAMPLING_INTERVAL_SECONDS == 5 else 'menit'}
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 45)

def channel_statistics(frame):
    numeric = frame.select_dtypes(include=np.number)
    result = numeric.describe(percentiles=[.01, .5, .99]).T
    result['missing'] = numeric.isna().sum()
    result['nonfinite_not_null'] = (numeric.notna() & ~np.isfinite(numeric)).sum()
    result['zeros'], result['negative'] = numeric.eq(0).sum(), numeric.lt(0).sum()
    result['unique'] = numeric.nunique(dropna=True)
    result['constant'] = result['unique'].le(1)
    return result[['count', 'mean', 'missing', 'nonfinite_not_null', 'zeros', 'negative',
                   'unique', 'constant', 'min', '1%', '50%', '99%', 'max', 'std']]

def inspect_actions(frame, parsed, interval):
    continuous = parsed.diff().dt.total_seconds().eq(interval)
    rows = []
    for column in ACTION_COLUMNS:
        if column not in frame:
            rows.append({'channel': column, 'status': 'MISSING'})
            continue
        values = pd.to_numeric(frame[column], errors='coerce')
        active = values.gt(0)
        valid_neighbor = continuous & values.notna() & values.shift().notna()
        rows.append({'channel': column, 'status': 'logged exposure; delivery/timing unverified',
                     'min': values.min(), 'max': values.max(), 'unique': values.nunique(),
                     'active_rows': int(active.sum()), 'active_recorded_minutes': active.sum() * interval / 60,
                     'adjacent_value_changes': int((valid_neighbor & ~np.isclose(values, values.shift(), equal_nan=False)).sum()),
                     'active_blocks': int((active & (~active.shift(fill_value=False) | ~continuous)).sum()),
                     'missing': int(values.isna().sum()),
                     'recorded_total_mL': values.sum(min_count=1) if column in DOSE_COLUMNS else np.nan,
                     'mean_setting': values.mean() if column in HELD_COLUMNS else np.nan})
    return pd.DataFrame(rows)

def inspect_sensor(frame, parsed, cultivation, interval):
    row = {'cultivation': cultivation, 'sampling_interval_seconds': interval,
           'pH_outside_0_14': int((frame['1_ph'].lt(0) | frame['1_ph'].gt(14)).sum()),
           'negative_DO': int(frame['1_do(mg/L)'].lt(0).sum()),
           'constant_aeration': frame['Aerator_RU (L/s)'].nunique() <= 1,
           'constant_lamp': frame['Lampu (PWM%)'].nunique() <= 1,
           'missing_action_columns': ', '.join(c for c in ACTION_COLUMNS if c not in frame)}
    valid_bod = frame[['1_bod(mg/L)', '1_cod(mg/L)']].dropna()
    row['BOD_equals_COD_over_2_pct'] = 100 * np.isclose(valid_bod.iloc[:, 0], valid_bod.iloc[:, 1] / 2, atol=1e-4, rtol=1e-4).mean()
    max_lux = 1050 if cultivation <= 10 else 1800
    row['lamp_formula_match_pct'] = 100 * np.isclose(frame['Lampu (lux)'], frame['Lampu (PWM%)'] / 100 * max_lux, atol=.02, rtol=1e-4).mean()
    row['biomass_formula_match_pct'] = 100 * np.isclose(frame['biomassa_estimasi(g/L)'], 1.2487 * frame['od680_estimasi(AU)'] + .0156, atol=1e-4, rtol=1e-4).mean()
    good = wm.observed_rows(frame, interval) & parsed.notna().to_numpy()
    lengths = wm.continuous_run_lengths(good, parsed, interval)
    for minutes in [60, 360, 1440]:
        row[f'eligible_{minutes}min_history_plus_one_step'] = int((lengths >= wm.duration_steps(minutes, interval) + 1).sum())
    pair = frame[DOSE_COLUMNS[:2]].dropna()
    active_pair = pair.loc[pair.gt(0).any(axis=1)]
    confounding = {'cultivation': cultivation,
                   'NaHCO3_molasses_profiles_identical': bool(np.allclose(pair.iloc[:, 0], pair.iloc[:, 1])) if len(pair) else None,
                   'rows_with_either_dose': len(active_pair),
                   'unequal_active_dose_rows': int((~np.isclose(active_pair.iloc[:, 0], active_pair.iloc[:, 1])).sum()),
                   'pump_profile_pearson': pair.iloc[:, 0].corr(pair.iloc[:, 1]) if pair.nunique().gt(1).all() else np.nan,
                   'incomplete_cultivation_per_README': cultivation in [3, 15],
                   'treatment_source_per_README': 'schedule/manual reconstructed' if cultivation <= 5 else 'treatment logs; originals not copied',
                   'nominal_lamp_max_lux_per_README': max_lux}
    target_complete = np.isfinite(frame.reindex(columns=QUALITY_TARGETS).to_numpy(dtype=float)).all(axis=1)
    row['eligible_60min_input_history_and_12_targets'] = int(((lengths >= wm.duration_steps(60, interval) + 1) & target_complete).sum())
    return row, confounding

def inspect_file(path):
    match = FILE_PATTERN.fullmatch(path.name)
    cultivation, kind, suffix = int(match[1]), match[2], match[3]
    interval = {'menit': 60, '5detik': 5, '10detik': 10}[suffix]
    expected = (12 if kind == 'sensor' else 6) if suffix == 'menit' else 1
    frame = pd.read_csv(path)
    parsed = pd.to_datetime(frame.timestamp, errors='coerce')
    times = pd.DatetimeIndex(parsed.dropna().sort_values().drop_duplicates())
    gaps = pd.Series(times).diff().dt.total_seconds()
    source = ORIGINAL_DIR / 'per_kultivasi' / f'K{cultivation:02d}' / path.name
    current_hash = wm.file_hash(path)
    n_data = pd.to_numeric(frame.n_data, errors='coerce') if 'n_data' in frame else None
    ids = pd.to_numeric(frame.kultivasi, errors='coerce') if 'kultivasi' in frame else None
    full = n_data.eq(expected) if n_data is not None else pd.Series(True, index=frame.index)
    summary = {'file': path.name, 'cultivation': cultivation, 'kind': kind, 'rows': len(frame), 'columns': len(frame.columns),
               'expected_interval_s': interval, 'modal_interval_s': gaps.mode().iloc[0] if gaps.notna().any() else np.nan,
               'start': times.min(), 'end': times.max(), 'span_days': (times.max() - times.min()).total_seconds() / 86400 if len(times) else np.nan,
               'invalid_timestamps': int(parsed.isna().sum()), 'duplicate_timestamps': int(parsed[parsed.notna()].duplicated().sum()),
               'duplicate_full_rows': int(frame.duplicated().sum()), 'timestamps_in_order': bool(parsed.dropna().is_monotonic_increasing),
               'gaps_over_expected_interval': int(gaps.gt(interval).sum()), 'intervals_under_expected': int(gaps.lt(interval).sum()),
               'missing_expected_slots': int((np.floor(gaps.dropna() / interval) - 1).clip(lower=0).sum()),
               'missing_cells': int(frame.isna().sum().sum()), 'full_sample_rows_or_bins': int(full.sum()),
               'partial_sample_bins': int((n_data.gt(0) & n_data.lt(expected)).sum()) if n_data is not None else 0,
               'unexpected_sample_bins': int((n_data.isna() | n_data.le(0) | n_data.gt(expected)).sum()) if n_data is not None else 0,
               'cultivation_id_mismatches': int((ids.isna() | ids.ne(cultivation)).sum()) if ids is not None else None,
               'quality_flags_available': 'status' in frame,
               'copy_matches_original': current_hash == wm.file_hash(source) if source.is_file() else None,
               'sha256': current_hash, 'source': str(source.relative_to(PROJECT_ROOT)) if source.is_file() else 'Original unavailable'}
    record = {'path': path, 'cultivation': cultivation, 'kind': kind, 'interval': interval,
              'columns': list(frame.columns), 'time': times, 'summary': summary, 'statistics': channel_statistics(frame)}
    if kind == 'sensor':
        record['quality'] = dq.inspect_quality(frame, parsed, cultivation, interval, QUALITY_TARGETS, ACTION_COLUMNS, QUALITY_CONFIG,
                                               DATASET_DIR / f'K{cultivation:02d}_sensor_menit.csv')
        record['actions'] = inspect_actions(frame, parsed, interval)
        record['readiness'], record['confounding'] = inspect_sensor(frame, parsed, cultivation, interval)
        display_data = frame.select_dtypes(include=np.number).copy()
        display_data.index = parsed
        display_data = display_data.loc[display_data.index.notna()].sort_index()
        record['plot_means'] = display_data.resample(PLOT_INTERVAL).mean()
        record['plot_doses'] = display_data[DOSE_COLUMNS].resample(PLOT_INTERVAL).sum(min_count=1)
        detail_end = display_data.index[0] + pd.Timedelta(minutes=DETAIL_MINUTES)
        record['detail'] = display_data.loc[display_data.index < detail_end, PROCESS_PLOT_COLUMNS]
        record['correlation'] = frame[PROCESS_PLOT_COLUMNS].corr()
    del frame
    gc.collect()
    return record

def timing_gaps(record):
    times = pd.Series(record['time'])
    gap = times.diff().dt.total_seconds()
    return pd.DataFrame({'previous': times.shift(), 'next': times, 'seconds': gap})[gap > record['interval']]

def action_statistics(record):
    return record['actions'].copy()

def plot_cultivation(record):
    if not PLOTTING_AVAILABLE:
        return
    means, doses = record['plot_means'], record['plot_doses']
    fig, axes = plt.subplots(3, 2, figsize=(13, 8), sharex=True)
    for axis, column, label in zip(axes.flat, PROCESS_PLOT_COLUMNS, ['pH', 'DO (mg/L)', 'Water temperature (°C)', 'Sensor COD (mg/L)']):
        axis.plot(means.index, means[column], linewidth=.9)
        axis.set_ylabel(label)
        axis.grid(alpha=.2)
    axes[2, 0].plot(means.index, means[HELD_COLUMNS[1]], linewidth=.9)
    axes[2, 0].set_ylabel('Lamp PWM (%)')
    second = axes[2, 0].twinx()
    second.plot(means.index, means[HELD_COLUMNS[0]], color='tab:orange', linewidth=.9)
    second.set_ylabel('Aeration (L/s)')
    for column in DOSE_COLUMNS:
        axes[2, 1].plot(doses.index, doses[column], label=column.split(' (')[0], linewidth=.9)
    axes[2, 1].set_ylabel(f'Recorded dose (mL / {PLOT_INTERVAL})')
    axes[2, 1].legend(fontsize=8)
    fig.suptitle(f'K{record["cultivation"]:02d} — {record["interval"]}s source; {PLOT_INTERVAL} display summaries')
    fig.autofmt_xdate()
    fig.tight_layout()
    buffer = BytesIO()
    fig.savefig(buffer, format='png', dpi=110, bbox_inches='tight')
    display(Image(data=buffer.getvalue()))
    plt.close(fig)
    fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
    detail = record['detail']
    for axis, column in zip(axes, PROCESS_PLOT_COLUMNS[:2]):
        axis.plot(detail.index, detail[column], marker='.', markersize=2, linewidth=.7)
        axis.set_ylabel(column)
        axis.grid(alpha=.2)
    fig.suptitle(f'K{record["cultivation"]:02d} — first {DETAIL_MINUTES} minutes at native file cadence ({record["interval"]}s); no interpolation')
    fig.tight_layout()
    buffer = BytesIO()
    fig.savefig(buffer, format='png', dpi=110, bbox_inches='tight')
    display(Image(data=buffer.getvalue()))
    plt.close(fig)

print('Sensor profile:', SAMPLING_INTERVAL_SECONDS, 'seconds | suffixes:', SUFFIXES)
print('Files are read one at a time; complete raw frames are not retained in records.')
print('Plotting available:', PLOTTING_AVAILABLE)
if not PLOTTING_AVAILABLE:
    print(PLOTTING_ERROR)
print('Quality targets:', QUALITY_TARGETS)
print('Priority channels:', PRIORITY_TARGETS)
print('Detailed plots:', QUALITY_PLOT_CULTIVATIONS, '| all cultivations retain 12-channel statistics/verdicts.')

## ***Per Data Evaluation Report***

### ***File inventory and source fidelity***

Inventory recognizes both native and minute files, and inspects the configured profile and cultivations. Exact hashes check each used-data copy against its source; they do not verify measurements. Native files have no `n_data`; for minute files, complete row coverage is not proof that all source readings were valid. No five-second MPPT observations are created.

In [ ]:
available_files = sorted(DATASET_DIR.glob('*.csv'))
unrecognized = [p.name for p in available_files if not FILE_PATTERN.fullmatch(p.name)]
if unrecognized:
    raise ValueError(f'Unrecognized files: {unrecognized}')
available_rows, files = [], []
for path in available_files:
    match = FILE_PATTERN.fullmatch(path.name)
    cultivation, kind, suffix = int(match[1]), match[2], match[3]
    selected = cultivation in CULTIVATIONS and suffix == SUFFIXES[kind]
    available_rows.append({'file': path.name, 'cultivation': cultivation, 'kind': kind, 'resolution': suffix,
                           'size_MiB': path.stat().st_size / 1024**2, 'selected': selected})
    if selected:
        files.append(path)
display(pd.DataFrame(available_rows))
expected_names = {f'K{k:02d}_{kind}_{suffix}.csv' for k in CULTIVATIONS for kind, suffix in SUFFIXES.items()}
missing_files = sorted(expected_names - {p.name for p in files})
if missing_files:
    raise FileNotFoundError(f'Missing selected-profile copies: {missing_files}')
if not files:
    raise FileNotFoundError('No files selected.')
records = [inspect_file(path) for path in wm.progress(files, desc='Inspect files sequentially', unit='file', enabled=True)]
inventory = pd.DataFrame([record['summary'] for record in records])
display(inventory[['file', 'rows', 'columns', 'start', 'end', 'expected_interval_s', 'modal_interval_s',
                   'missing_cells', 'duplicate_timestamps', 'gaps_over_expected_interval',
                   'full_sample_rows_or_bins', 'partial_sample_bins', 'copy_matches_original']])
display(inventory.groupby('kind').agg(files=('file', 'count'), rows=('rows', 'sum'), missing_cells=('missing_cells', 'sum')))
print('Selected profile:', SAMPLING_INTERVAL_SECONDS, 'seconds | inspected files:', len(records))
print('Copy mismatches:', inventory.loc[inventory.copy_matches_original.eq(False), 'file'].tolist())

### ***Channel meanings and known provenance limits***

These descriptions come from the original dataset README, not from correlations. Verify physical calibration separately. `1_temp_air(C)` uses the Indonesian word *air* and represents **water temperature**. There is no dedicated NH3-temperature channel in these CSVs.

The README says K1–K5 treatment values were reconstructed from daily schedules/manual doses, and K6–K16 from treatment logs. The copied CSVs do not contain the original command histories or measurement-quality flags. Thus they can describe logged exposures, but do not certify delivered treatments or support arbitrary new RL actions.

In [ ]:
provenance_rows = []
sensor_columns = sorted(set().union(*(set(r['columns']) for r in records if r['kind'] == 'sensor')))
for column in sensor_columns:
    units_match = re.search(r'\(([^()]*)\)$', column)
    units = units_match[1] if units_match else ('dimensionless' if column == '1_ph' else 'not in header')
    if column in ['timestamp', 'kultivasi', 'hari_ke', 'n_data']:
        role, status = 'index/metadata', 'not a biological target; n_data is row coverage'
    elif column in DOSE_COLUMNS:
        role, status = 'candidate treatment', 'recorded mL per source row/bin; stock/delivery/timing unverified'
    elif column in HELD_COLUMNS:
        role, status = 'candidate treatment', 'source setting/flow; minute aggregates are means; delivered control unverified'
    elif column == 'biomassa_estimasi(g/L)':
        role, status = 'added output-only proxy target', 'interpolated OD-derived biomass with synthetic fluctuations; future-anchor leakage; not independent biomass/noise truth'
    elif column in ESTIMATED_COLUMNS:
        role, status = 'excluded estimated auxiliary field', 'estimated from interpolated lab OD; future-anchor leakage risk'
    elif column == 'Lampu (lux)':
        role, status = 'derived actuator representation', 'calculated from PWM; use PWM instead of duplicating this action'
    elif column == '1_tss(mg/L)':
        role, status = 'provisional process measurement', 'added output-only proxy target; K1–K9 estimated; newer calibration still unverified'
    elif column == '1_bod(mg/L)':
        role, status = 'added output-only BOD proxy target', 'README describes firmware COD/2; numerical disagreement is audited; not lab BOD'
    elif column == '1_temp_air(C)':
        role, status = 'process observation', 'water temperature; physical calibration unverified'
    elif column == '2_light(Lux)':
        role, status = 'measured light observation', 'distinct from commanded lamp PWM and nominal lux'
    else:
        role, status = 'sensor observation candidate', 'header/README mapping; calibration and independence unverified'
    provenance_rows.append({'column': column, 'header_unit': units, 'role': role, 'provenance_limit': status, 'prediction_target': column in QUALITY_TARGETS, 'priority': column in PRIORITY_TARGETS})
display(pd.DataFrame(provenance_rows))
print('Provenance reference:', SOURCE_README.relative_to(PROJECT_ROOT), '| exists:', SOURCE_README.is_file())
lab_files = list(DATASET_DIR.rglob('*.xlsx')) + list(DATASET_DIR.rglob('*.xls'))
print('Actual lab workbooks among used-data copies:', [p.name for p in lab_files] or 'none')
print('Original treatment-log files in used-data copies:', [p.name for p in DATASET_DIR.rglob('*') if p.is_file() and 'treatment' in p.name.lower()] or 'none')

### ***Individual file reports***

Report cadence, gaps, quality availability, native-unit statistics, and treatment variation at each file's actual resolution. Zero and negative values are reported without deletion. Pump totals sum logged per-row/bin volumes once. Active rows and blocks are not independently verified commands. Plot summaries are for display only; the detail plot retains the file's native readings.

In [ ]:
for record in records:
    display(Markdown(f'#### ***{record["path"].name}***'))
    display(pd.Series(record['summary'], name='Observed file facts').to_frame())
    gaps = timing_gaps(record)
    if len(gaps):
        display(gaps.head(MAX_GAP_ROWS))
    else:
        print('No timestamp gaps longer than the expected', record['interval'], 'seconds.')
    if record['kind'] == 'sensor' or SHOW_MPPT_CHANNEL_STATISTICS:
        display(record['statistics'])
    if record['kind'] == 'sensor':
        display(action_statistics(record))
        if SHOW_LEGACY_TREND_PLOTS:
            plot_cultivation(record)
        if SHOW_SENSOR_CORRELATION:
            display(record['correlation'].round(3))

### ***Cross-cultivation treatment coverage and confounding***

Compare observed treatment variation across cultivations. An actuator that stays constant within a cultivation provides little evidence about the response to changing it. If two pump profiles are identical, this dataset cannot separate their individual effects in that run. Inoculum addition can increase OD without demonstrating biological reproduction.

Dose totals below are **recorded volumes**. Aerator and lamp means are settings, not independently measured energy costs. Do not use observed minima and maxima as safe RL bounds.

In [ ]:
sensor_records = sorted((r for r in records if r['kind'] == 'sensor'), key=lambda r: r['cultivation'])
treatment_tables = []
confounding_rows = []
for record in sensor_records:
    table = action_statistics(record)
    table.insert(0, 'cultivation', record['cultivation'])
    treatment_tables.append(table)
    confounding_rows.append(record['confounding'])
treatment_coverage = pd.concat(treatment_tables, ignore_index=True)
display(treatment_coverage)
display(pd.DataFrame(confounding_rows))
display(treatment_coverage.pivot(index='cultivation', columns='channel', values='unique'))

### ***Sensor / MPPT timestamp alignment and electrical-data scope***

Compare exact timestamps at their actual cadences: native sensor is 5 seconds and MPPT is 10 seconds; minute aggregates are both 60 seconds. Partial timestamp overlap is expected in the native profile. No MPPT values are forward-filled or interpolated into sensor rows. Clock overlap does not prove synchronization or that the electrical load belongs only to cultivation equipment. Electrical readings remain outside the primary world-model features.

In [ ]:
alignment_rows = []
for record in sensor_records:
    k = record['cultivation']
    mppt = next((r for r in records if r['cultivation'] == k and r['kind'] == 'rasp_mppt'), None)
    if mppt is None:
        alignment_rows.append({'cultivation': k, 'status': 'MPPT absent'})
        continue
    sensor_times, mppt_times = record['time'], mppt['time']
    matched = sensor_times.intersection(mppt_times)
    alignment_rows.append({'cultivation': k, 'sensor_interval_s': record['interval'], 'MPPT_interval_s': mppt['interval'],
                           'matched_timestamps': len(matched), 'sensor_only_timestamps': len(sensor_times.difference(mppt_times)),
                           'MPPT_only_timestamps': len(mppt_times.difference(sensor_times)),
                           'sensor_timestamp_match_pct': 100 * len(matched) / len(sensor_times) if len(sensor_times) else np.nan,
                           'same_timestamp_set': sensor_times.equals(mppt_times)})
display(pd.DataFrame(alignment_rows))
electrical_cols = ['pv_voltage', 'pv_current', 'pv_power', 'battery_voltage', 'battery_charge_current',
                   'battery_charge_power', 'battery_remaining', 'load_current', 'load_power']
electrical_rows = []
for record in records:
    if record['kind'] != 'rasp_mppt':
        continue
    stats = record['statistics']
    for column in electrical_cols:
        if column in stats.index:
            values = stats.loc[column]
            electrical_rows.append({'cultivation': record['cultivation'], 'column': column,
                                    'min': values['min'], 'mean': values['mean'], 'max': values['max'],
                                    'missing': values['missing'], 'unique': values['unique'],
                                    'unit': 'not in header; verify source calibration'})
display(pd.DataFrame(electrical_rows))

### ***Derived-channel consistency and contiguous-history availability***

Formula agreement indicates dependence, not measurement accuracy. Count histories in **physical minutes**, with one next-step target at the selected sensor resolution. The strict mask matches world-model feature completeness, timestamp continuity, minute `n_data=12`, and explicit OK flags where present. Missing quality flags remain unresolved; this notebook does not impute observations or choose the history hyperparameter from held-out outcomes.

In [ ]:
readiness = pd.DataFrame([record['readiness'] for record in sensor_records])
display(readiness)
schema_rows = []
for kind in ['sensor', 'rasp_mppt']:
    selected = [r for r in records if r['kind'] == kind]
    reference = selected[0]['columns'] if selected else []
    for record in selected:
        schema_rows.append({'file': record['path'].name,
                            'same_ordered_schema_as_first_of_kind': record['columns'] == reference,
                            'missing_columns': ', '.join(set(reference) - set(record['columns'])),
                            'extra_columns': ', '.join(set(record['columns']) - set(reference))})
display(pd.DataFrame(schema_rows))

### ***Noise and outlier methodology what these screens can establish***

All **12 prediction targets** are assessed at the selected source cadence. Per-channel values retain their units; never compare COD variance numerically with pH variance. Population level variance uses `ddof=0`; the within-minute boxplot uses sample variance (`ddof=1`). Within/between-minute decomposition uses only bins with exactly 12 finite, unique, regularly spaced five-second readings and converts the within-bin variance to population variance.

| Diagnostic | Meaning and limitation |
|---|---|
| Value distribution | Full-file levels; boxes show 25th–75th percentiles, line median, whiskers 5th–95th. Extreme values remain in statistics and source files. |
| Within-minute variance | Distribution of variability among the 12 readings in complete minutes. It can contain real responses as well as noise. |
| Within-minute share | `mean(population variance within bins) / (within + variance of bin means)`. Denominator concerns the same complete equal-size bins. Undefined for zero variance. |
| Robust fast-change scale | `MAD(adjacent changes) / 0.67449`; changes never cross gaps, invalid timestamps, or missing/flagged observations. |
| Noise-scale proxy | Robust change scale / sqrt(2). It estimates measurement noise only under locally constant signal and independent equal-variance errors. No such assumption is verified here. |
| Outlier candidate | Deviation from a centered 3-minute local median greater than 6 robust residual scales. Centered smoothing is EDA only and must not become future-informed training preprocessing. A zero MAD makes this screen unavailable rather than proving no outliers. |
| Isolated spike | An outlier candidate with valid neighbours, opposite successive changes, and return within 25% of the larger jump. Real short responses can still qualify. |
| Flatline / quantization | Exact-value runs, ties, zero frequency and min/max pileup. They can indicate stability, rounding, freeze or clipping; hardware limits are unverified. |
| Drift / regime change | First-versus-last 10% median shift in IQR units. Cultivation evolution is not automatically sensor drift. |
| Welch spectrum | One longest contiguous finite segment per channel, capped at 17,280 samples; linear detrending. Fast power (period <=60 s) is not automatically sensor noise. No gap interpolation. |
| Minute consistency | Compare independently supplied complete-minute means with native-bin means at identical timestamps. Configured tolerances reflect the observed CSV output precision (0.1 for COD/BOD/NH3, 0.01 for other sensors, 0.0001 for biomass), with a small numerical allowance; these are aggregation checks, not calibration limits. |

Review triggers (10-minute flatline, >=0.1% isolated candidates, >=20% within-minute variance share) are **declared exploratory settings**, not validated process acceptance limits or automatic deletion rules. The pH 0–14 interval is a conventional screening range, not the reactor's approved operating band; temperature has no invented operating bound. Missing/duplicate timestamps and quality flags are reported separately.

The source README describes TSS/BGA brushing and COD cleaning every 2 hours for 30 seconds. This is a possible explanation to investigate for brief disturbances; no synchronized cleaning-event log is available here. A spike within 60 seconds of a logged treatment is merely temporally associated.

Method references: [SciPy MAD](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.median_abs_deviation.html), [SciPy Welch PSD](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.welch.html), and [NIST on investigating outliers](https://www.itl.nist.gov/div898/handbook/eda/section3/eda35h.htm). The numerical thresholds above are project screens, not thresholds taken from these references.

In [ ]:
quality_metrics = pd.concat([r['quality']['metrics'] for r in sensor_records], ignore_index=True)
channel_quality = dq.channel_verdicts(quality_metrics, QUALITY_CONFIG)
roles = {11: 'train', 12: 'train', 13: 'train', 14: 'validation', 16: 'reserved test'}
channel_quality['split_role'] = channel_quality.cultivation.map(roles).fillna('historical / not current training split')
print('Evaluated targets:', len(QUALITY_TARGETS), '| cultivations:', len(sensor_records))
display(pd.DataFrame({'target_order': QUALITY_TARGETS, 'priority': [t in PRIORITY_TARGETS for t in QUALITY_TARGETS]}))
display(pd.Series({k: v for k, v in QUALITY_CONFIG.items() if k not in ['figure_dir', 'show_plots']}, name='Declared diagnostic settings').to_frame())
priority_columns = ['cultivation', 'split_role', 'label', 'provenance', 'level_std', 'step_robust_sigma',
                    'noise_proxy_to_level_std', 'within_minute_variance_share_pct', 'isolated_spike_pct',
                    'longest_flat_minutes', 'screen_verdict']
display(channel_quality.loc[channel_quality.priority, priority_columns])
print('Priority is a group; no channel weights or priority ranking are inferred.')

### ***All-target value and variance distributions across cultivations***

The first figure compares level distributions; the second compares within-minute variances, with a symmetric-log axis so zero-variance minutes remain visible. Each panel has its own units and scale. Heatmaps expose timing-independent descriptive warning signs across every cultivation and every target. Large overall variance may be a useful treatment/growth signal; it is not itself evidence of noise.

In [ ]:
if PLOTTING_AVAILABLE:
    dq.plot_distributions(sensor_records, QUALITY_TARGETS, QUALITY_CONFIG)
    if SAMPLING_INTERVAL_SECONDS == 5:
        dq.plot_distributions(sensor_records, QUALITY_TARGETS, QUALITY_CONFIG, variance=True)
    else:
        print('Within-minute native variance cannot be reconstructed from minute means; select the 5-second profile.')
    dq.plot_quality_heatmaps(channel_quality, QUALITY_TARGETS, QUALITY_CONFIG)
else:
    print('Plots unavailable:', PLOTTING_ERROR)

### ***Native detail, change distributions, and spectra for the training/validation cultivations***

Every detailed figure contains **all 12 targets**, with priority labels on the native traces. Windows surround the earliest detected isolated-spike candidate among the priority channels, or use the first available interval when none is found. This selection is disclosed and is not a model-performance comparison. Native traces, EDA local medians, and complete-minute means are drawn together; averaging changes the observation target and can conceal fast responses.

Histograms standardize changes by their robust scale when it is positive. Counts outside the displayed +/-12-scale range are explicitly reported rather than discarded. PSD panels state the number of contiguous observations analyzed. Investigate periodic features against actual cleaning/actuator logs before attributing them to instrumentation.

In [ ]:
for record in wm.progress([r for r in sensor_records if r['cultivation'] in QUALITY_PLOT_CULTIVATIONS],
                          desc='Render 12-target diagnostics', unit='cultivation'):
    display(Markdown(f"#### ***K{record['cultivation']:02d}: all-target diagnostics***"))
    fields = ['label', 'priority', 'missing_pct', 'range_screen_count', 'unique_values',
              'unchanged_adjacent_pct', 'longest_flat_minutes', 'lag1_correlation',
              'first_last_median_shift_IQR', 'local_outlier_pct', 'isolated_spike_candidates',
              'spikes_near_logged_action_60s', 'spectral_fast_power_pct']
    display(record['quality']['metrics'][fields])
    if PLOTTING_AVAILABLE:
        dq.plot_native_diagnostics(record, QUALITY_TARGETS, QUALITY_CONFIG)

### ***Native-versus-minute aggregation consistency and suspected event examples***

Only complete, regular native minutes are compared with supplied `n_data=12` minute means. Report pair counts and absolute errors in each target's own units. An unmatched or incomplete bin is not silently filled. Agreement verifies an aggregation relationship, not biological validity or absence of sensor noise. Short spike examples include neighbours and nearby logged-action flags for review; candidates remain unchanged in the dataset.

In [ ]:
aggregation_columns = ['cultivation', 'label', 'complete_native_minutes', 'within_minute_variance_mean',
                       'between_minute_variance', 'within_minute_variance_share_pct', 'provided_minute_pairs',
                       'provided_mean_MAE', 'provided_mean_p99_abs_difference', 'provided_mean_within_tolerance_pct']
display(channel_quality[aggregation_columns])
example_frames = [r['quality']['examples'] for r in sensor_records if not r['quality']['examples'].empty]
spike_examples = pd.concat(example_frames, ignore_index=True) if example_frames else pd.DataFrame(
    columns=['cultivation', 'target', 'timestamp', 'previous', 'value', 'next', 'local_median', 'near_logged_action_60s'])
display(spike_examples.head(60))
print('Retained candidate examples:', len(spike_examples), '| up to', QUALITY_CONFIG['max_event_examples_per_target'], 'per target per cultivation; counts in metrics use all rows.')

### ***Per-cultivation and per-channel verdicts***

These verdicts distinguish structural usability, statistical review candidates and unresolved measurement provenance. **NO SCREEN TRIGGER is not a clean-sensor certificate.** REVIEW calls for inspection, not automatic removal. BOD/biomass/older TSS can look exceptionally smooth because they are derived; that does not make them independently reliable outcomes. Cultivation-level judgments retain incomplete-run, action-confounding and source-fidelity concerns from the earlier sections.

In [ ]:
verdict_rows = []
for record in sensor_records:
    k = record['cultivation']
    q = channel_quality.loc[channel_quality.cultivation.eq(k)]
    priority = q.loc[q.priority]
    summary = record['summary']
    structure_issues = []
    if not record['readiness']['eligible_60min_input_history_and_12_targets']:
        structure_issues.append('no complete 60-minute input history plus 12-target label')
    if summary['invalid_timestamps'] or summary['duplicate_timestamps'] or not summary['timestamps_in_order']:
        structure_issues.append('invalid/duplicate/unordered timestamps')
    if not q.column_present.all() or q.usable_rows.eq(0).any():
        structure_issues.append('target absent or unusable')
    if summary['copy_matches_original'] is False:
        structure_issues.append('source-copy mismatch')
    reasons = []
    if structure_issues:
        reasons.extend(structure_issues)
    if summary['gaps_over_expected_interval']:
        reasons.append('segment timestamp gaps; never interpolate history across them')
    if len(priority.loc[priority.isolated_spike_pct.ge(QUALITY_CONFIG['spike_review_pct'])]):
        reasons.append('review isolated spikes in priority channels')
    fast = priority.loc[priority.within_minute_variance_share_pct.ge(QUALITY_CONFIG['within_variance_review_pct']), 'label'].tolist()
    if fast:
        reasons.append('large within-minute share in ' + ', '.join(fast))
    if q.longest_flat_minutes.ge(QUALITY_CONFIG['flatline_review_minutes']).any():
        reasons.append('inspect long flatlines / rounding / boundary pileup')
    if k in [3, 15]:
        reasons.append('incomplete cultivation per README')
    if record['readiness']['constant_aeration']:
        reasons.append('constant aeration limits treatment-response identification')
    if record['confounding']['NaHCO3_molasses_profiles_identical']:
        reasons.append('pump profiles confounded')
    verdict_rows.append({'cultivation': k, 'split_role': roles.get(k, 'historical'),
                         'structural_verdict': 'REQUIRES STRUCTURAL REVIEW' if structure_issues else 'usable contiguous segments exist; measurement validity unverified',
                         'targets_assessed': len(q), 'channels_with_review_trigger': int(q.screen_verdict.ne('NO SCREEN TRIGGER').sum()),
                         'priority_channels_with_review_trigger': int(priority.screen_verdict.ne('NO SCREEN TRIGGER').sum()),
                         'eligible_60min_history_plus_target': record['readiness']['eligible_60min_input_history_and_12_targets'],
                         'quality_flags_available': summary['quality_flags_available'],
                         'noise_verdict': '; '.join(reasons) or 'No configured numerical trigger; calibration and noise identity remain unverified',
                         'provenance_verdict': 'Biomass is interpolated; BOD provisional/derived; older TSS estimated; independent outcome validation absent',
                         'cadence_verdict': 'Compare 5s and 60s on K14 at common real-time horizons; do not select filters/cadence using K16'})
cultivation_quality = pd.DataFrame(verdict_rows)
display(cultivation_quality)
for k in sorted(channel_quality.cultivation.unique()):
    display(Markdown(f'#### ***K{k:02d}: verdict for every prediction target***'))
    display(channel_quality.loc[channel_quality.cultivation.eq(k),
            ['label', 'priority', 'screen_verdict', 'reason', 'provenance']])
training_quality = channel_quality.loc[channel_quality.cultivation.isin([11, 12, 13]) & channel_quality.priority]
if not training_quality.empty:
    display(Markdown('#### ***Training-cultivation evidence for the five priority channels***'))
    display(training_quality[['cultivation', 'label', 'within_minute_variance_share_pct', 'noise_proxy_to_level_std',
                              'isolated_spike_pct', 'longest_flat_minutes', 'screen_verdict']])
    fast_training = training_quality.loc[training_quality.within_minute_variance_share_pct.ge(QUALITY_CONFIG['within_variance_review_pct'])]
    display(Markdown(f"**Verdict:** {len(fast_training)}/{len(training_quality)} training-cultivation/priority-channel combinations exceed the declared within-minute variability review threshold. This is evidence to test aggregation sensitivity, not proof that those fluctuations are measurement noise. Use the native five-second files as the preserved source; evaluate any causal filter or 60-second aggregation as a separately labeled training recipe."))
print('No values were removed, replaced, clipped, smoothed for training, or written back to dataset CSVs.')

### ***Quality report exports and reproducibility***

Exports preserve all 12-channel statistics, explicit proxy provenance, heuristic settings, source hashes, per-channel and per-cultivation verdicts, aggregation consistency, and bounded candidate examples. CSVs/figures belong to this evaluation report; they are not replacement training data. No training or model evaluation is performed here.

In [ ]:
if EXPORT_QUALITY_REPORTS:
    QUALITY_REPORT_DIR.mkdir(parents=True, exist_ok=True)
    exports = {'quality_metrics': quality_metrics, 'channel_verdicts': channel_quality,
               'cultivation_verdicts': cultivation_quality, 'spike_examples': spike_examples,
               'aggregation_consistency': channel_quality[aggregation_columns]}
    for name, table in exports.items():
        table.to_csv(QUALITY_REPORT_DIR / f'{name}.csv', index=False)
    report_manifest = {
        'sensor_interval_seconds': SAMPLING_INTERVAL_SECONDS,
        'targets': QUALITY_TARGETS, 'priority_targets': PRIORITY_TARGETS,
        'cultivations': CULTIVATIONS, 'detailed_plot_cultivations': QUALITY_PLOT_CULTIVATIONS,
        'settings': QUALITY_CONFIG, 'analysis_backend_sha256': wm.file_hash(dq.__file__),
        'versions': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__},
        'sources': inventory[['file', 'sha256', 'rows', 'expected_interval_s']].to_dict('records'),
        'minute_comparison_sources': [{'file': path.name, 'sha256': wm.file_hash(path)}
                                      for k in CULTIVATIONS
                                      if (path := DATASET_DIR / f'K{k:02d}_sensor_menit.csv').is_file()]
                                      if SAMPLING_INTERVAL_SECONDS == 5 else [],
        'source_documentation': {'file': str(SOURCE_README.relative_to(PROJECT_ROOT)),
                                 'sha256': wm.file_hash(SOURCE_README) if SOURCE_README.is_file() else None},
        'limitations': ['Statistical screens do not isolate sensor noise from true dynamics',
                        'No verified calibration, noise ground truth or operating limits',
                        'Interpolated/derived outcomes are not independent measurements',
                        'Local centered median is EDA only and is not exported as model input',
                        'Held-out cultivation EDA does not authorize selecting preprocessing on test outcomes'],
    }
    (QUALITY_REPORT_DIR / 'manifest.json').write_text(json.dumps(report_manifest, indent=2), encoding='utf-8')
    print('Saved quality reports and figures:', QUALITY_REPORT_DIR)
else:
    print('Exports disabled; reports remain displayed in this notebook.')

### ***EDA findings and decisions still needed***

This report describes all copied cultivations. It does not choose hyperparameters from held-out cultivation outcomes, fill gaps, refit sensors, or authorize controller actions.

The current training scope retains the original nine outputs and adds biomass estimate, BOD, and TSS (12 total). The three added fields are output-only proxy labels; they are explicitly assessed here. For independent biological validation, use actual laboratory outcomes rather than interpolated/derived labels. Calculated lamp lux remains excluded. Older estimated TSS and BOD formula discrepancies need source investigation. Keep cultivation boundaries and equipment regimes when making splits. Retain actual lab measurements separately for biological evaluation.

The available action candidates are NaHCO3 dose, molasses dose, inoculum dose, aeration, and lamp PWM. Resolve dose concentrations/calibration, command-versus-delivery timing, one-time inoculum use, and actuator variation before treating them as RL controls. CO2 injection, water/dilution, and NH3 dosing are not directly logged under those meanings. Future logged actions may condition historical prediction, but must not become future sensor inputs or evidence that a new policy works.


In [ ]:
constant_air = readiness.loc[readiness.constant_aeration, 'cultivation'].tolist()
identical_pumps = [row['cultivation'] for row in confounding_rows if row['NaHCO3_molasses_profiles_identical']]
bod_disagreement = readiness.loc[readiness.BOD_equals_COD_over_2_pct.lt(100), 'cultivation'].tolist()
display(Markdown(
    f"- Sensor profile: **{SAMPLING_INTERVAL_SECONDS} seconds**; MPPT retains its selected file cadence.\n"
    f"- Inspected **{len(inventory)} files**, including **{int(inventory.loc[inventory.kind.eq('sensor'), 'rows'].sum()):,} sensor rows**.\n"
    f"- Copy fidelity: **{int(inventory.copy_matches_original.eq(True).sum())}/{len(inventory)}** match source files.\n"
    f"- Missing table cells: **{int(inventory.missing_cells.sum()):,}**; this does not establish verified acquisition.\n"
    f"- Partial aggregation bins: **{int(inventory.partial_sample_bins.sum()):,}**.\n"
    f"- Files with explicit quality flags: **{int(inventory.quality_flags_available.sum())}/{len(inventory)}**.\n"
    f"- Cultivations with constant aeration: **{constant_air}**.\n"
    f"- Identical NaHCO3/molasses profiles: **{identical_pumps}**.\n"
    f"- BOD formula discrepancies: **{bod_disagreement}**.\n"
    f"- Actual lab workbooks among copied files: **{len(lab_files)}**."
))